[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Digital-AI-Finance/Introduction-to-Machine-Learning-notebooks/blob/master/forest_basics.ipynb)

# Random forests, in pictures

Run each cell with **Shift and Enter**, from the top. One cell sets everything
up; after that each cell is a single line with a single number in it. Four
questions are put along the way, and each answer is one click away.

The companion notebook, **Decision trees, in pictures**, ends on a single tree
that follows the noise in the rows it was fitted on. That is the problem taken
up here.

## What is a random forest?

Grow several hundred trees, give each one a slightly different view of the
data, and let them vote. Each tree is grown on its own resample of the rows and
is allowed to look at only a random handful of the columns at each cut, so no
two trees come out the same.

One tree follows the noise in whatever rows it was handed. The noise is
different in each resample, so it pulls each tree in a different direction, and
averaging over the trees cancels much of it. What the trees agree on is the
signal, because the signal is the part that survives the resampling.

In [ ]:
# Run this cell once. Every cell after it is one line.
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, make_moons
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from matplotlib.colors import LinearSegmentedColormap

INK, AMBER, GREY = "#1e3a5f", "#b45309", "#64748b"
TINT = ["#dfe4ec", "#fbeddc"]
SHADE = LinearSegmentedColormap.from_list("arc", ["#c9d3e4", "white", "#f7dcb8"])

ARC_X, ARC_Y = make_moons(n_samples=400, noise=0.30, random_state=0)
FIT_X, HELD_X, FIT_Y, HELD_Y = train_test_split(
    ARC_X, ARC_Y, test_size=0.3, random_state=0, stratify=ARC_Y)

iris = load_iris()
IRIS_X, IRIS_Y = iris.data, iris.target
COLUMNS = [n.replace(" (cm)", "") for n in iris.feature_names]


def _points(ax, data, label, size=14):
    for k, mark in enumerate("os"):
        ax.scatter(data[label == k, 0], data[label == k, 1], marker=mark,
                   s=size, color=[INK, AMBER][k], zorder=3)


def _grid():
    pad = 0.4
    gx, gy = np.meshgrid(
        np.linspace(ARC_X[:, 0].min() - pad, ARC_X[:, 0].max() + pad, 260),
        np.linspace(ARC_X[:, 1].min() - pad, ARC_X[:, 1].max() + pad, 260))
    return gx, gy, np.c_[gx.ravel(), gy.ravel()]


def _zones(ax, model):
    """A yes or a no at every point on the plane."""
    gx, gy, grid = _grid()
    zone = model.predict(grid).reshape(gx.shape)
    ax.contourf(gx, gy, zone, levels=[-0.5, 0.5, 1.5], colors=TINT)
    ax.contour(gx, gy, zone, levels=[0.5], colors=AMBER, linewidths=1.0)
    ax.set_xticks([])
    ax.set_yticks([])


def _share(ax, value, gx, gy):
    """The share of the trees voting for arc 1, white where they split evenly."""
    ax.contourf(gx, gy, value, levels=np.linspace(0, 1, 25), cmap=SHADE)
    ax.contour(gx, gy, value, levels=[0.5], colors=AMBER, linewidths=1.2)
    ax.set_xticks([])
    ax.set_yticks([])


def _resample(seed):
    """A bootstrap sample: n rows drawn from n, with replacement."""
    rows = np.random.RandomState(seed).randint(0, len(FIT_X), len(FIT_X))
    return FIT_X[rows], FIT_Y[rows]


def idea():
    """Three trees on three resamples, and the majority of the three."""
    fig, grid_axes = plt.subplots(2, 2, figsize=(8.4, 6.0))
    axes = grid_axes.ravel()
    trees = []
    for k, ax in enumerate(axes[:3]):
        sx, sy = _resample(k)
        tree = DecisionTreeClassifier(random_state=0).fit(sx, sy)
        trees.append(tree)
        _zones(ax, tree)
        _points(ax, sx, sy, size=8)
        ax.set_title("tree %d" % (k + 1), fontsize=10)

    gx, gy, grid = _grid()
    wood = RandomForestClassifier(n_estimators=50, random_state=0).fit(
        FIT_X, FIT_Y)
    _share(axes[3], wood.predict_proba(grid)[:, 1].reshape(gx.shape), gx, gy)
    _points(axes[3], ARC_X, ARC_Y, size=8)
    axes[3].set_title("the share of fifty voting arc 1", fontsize=10)
    plt.tight_layout()
    plt.show()


def one_tree(seed):
    """One tree, grown on one resample of the rows."""
    sx, sy = _resample(seed)
    tree = DecisionTreeClassifier(random_state=0).fit(sx, sy)
    fig, ax = plt.subplots(figsize=(5.2, 3.4))
    _zones(ax, tree)
    _points(ax, sx, sy)
    ax.set_title("seed %d: %d leaves, %.3f of the rows held back"
                 % (seed, tree.get_n_leaves(), tree.score(HELD_X, HELD_Y)))
    plt.show()


def forest(trees):
    """That many trees, each on its own resample, answering by majority."""
    wood = RandomForestClassifier(n_estimators=trees, random_state=0).fit(
        FIT_X, FIT_Y)
    gx, gy, grid = _grid()
    fig, ax = plt.subplots(figsize=(5.2, 3.4))
    _share(ax, wood.predict_proba(grid)[:, 1].reshape(gx.shape), gx, gy)
    _points(ax, ARC_X, ARC_Y)
    ax.set_title("%d tree%s: %.3f of the rows held back"
                 % (trees, "" if trees == 1 else "s", wood.score(HELD_X, HELD_Y)))
    plt.show()


def votes(row):
    """How ten trees split over one row held back, and where the row sits."""
    wood = RandomForestClassifier(n_estimators=10, random_state=0).fit(
        FIT_X, FIT_Y)
    point = HELD_X[[row]]
    each = np.array([int(t.predict(point)[0]) for t in wood.estimators_])
    fig, (a, b) = plt.subplots(1, 2, figsize=(9.5, 3.2))
    a.bar(range(1, 11), np.ones(10), color=[[INK, AMBER][v] for v in each])
    for k, v in enumerate(each):
        a.text(k + 1, 0.5, "arc %d" % v, ha="center", va="center",
               rotation=90, color="white", fontsize=9)
    a.set_xticks(range(1, 11))
    a.set_yticks([])
    a.set_xlabel("tree")
    a.set_title("%d of the ten say arc 1" % each.sum(), fontsize=10)
    _zones(b, wood)
    _points(b, ARC_X, ARC_Y, size=10)
    b.scatter(point[0, 0], point[0, 1], marker="*", s=520, color="white",
              edgecolor=INK, linewidth=1.8, zorder=5)
    b.set_title("row %d, and it truly belongs to arc %d"
                % (row, HELD_Y[row]), fontsize=10)
    plt.tight_layout()
    plt.show()
    print("the ten trees: %s" % each.tolist())
    print("the forest answers %d, and the truth is %d"
          % (wood.predict(point)[0], HELD_Y[row]))


def steadiness():
    """What adding trees does to the share the forest gets right."""
    sizes = [1, 2, 3, 5, 10, 20, 50, 100, 200]
    got = [RandomForestClassifier(n_estimators=n, random_state=0)
           .fit(FIT_X, FIT_Y).score(HELD_X, HELD_Y) for n in sizes]
    lone = DecisionTreeClassifier(random_state=0).fit(FIT_X, FIT_Y).score(
        HELD_X, HELD_Y)
    fig, ax = plt.subplots(figsize=(6.0, 3.4))
    ax.semilogx(sizes, got, color=INK, lw=1.6, marker="o", ms=4)
    ax.axhline(lone, color=GREY, lw=1.2, ls="--")
    ax.set_xlabel("number of trees")
    ax.set_ylabel("share of the rows held back")
    ax.set_xticks(sizes)
    ax.set_xticklabels([str(n) for n in sizes])
    ax.set_ylim(min(lone, min(got)) - 0.014, max(got) + 0.008)
    ax.text(1, lone + 0.003, "one tree, all the rows: %.3f" % lone,
            fontsize=9, color=GREY)
    ax.text(sizes[-1], got[-1] + 0.003, "forest", fontsize=9, color=INK,
            ha="right")
    plt.show()


def importance(trees):
    """How much each of the four iris measurements is leaned on."""
    wood = RandomForestClassifier(n_estimators=trees, random_state=0).fit(
        IRIS_X, IRIS_Y)
    share = wood.feature_importances_
    fig, ax = plt.subplots(figsize=(6.0, 3.0))
    ax.barh(COLUMNS, share, color=[GREY, GREY, INK, INK])
    ax.set_xlabel("share of the impurity removed")
    ax.set_title("%d tree%s" % (trees, "" if trees == 1 else "s"), fontsize=10)
    for k, v in enumerate(share):
        ax.text(v + 0.01, k, "%.3f" % v, va="center", fontsize=9, color=INK)
    ax.set_xlim(0, max(share) * 1.25)
    plt.show()

## 1. Many trees, one vote

In [ ]:
idea()

Three trees on three resamples of the same 280 rows. Each answers yes or no at
every point, each draws a different boundary, and each fences off a few single
points out in the wrong arc.

The fourth panel is the share of fifty such trees voting for arc 1. Full colour
is where all fifty agree; white is where they split evenly; and the amber line
is where the vote crosses a half, which is the answer the forest gives. The
hard edge of a single tree is replaced by a gradient, and the width of that
gradient is where the trees disagreed.

## 2. One tree moves when the rows move

400 points in two interleaving arcs, 280 used to fit and 120 held back. A
resample draws 280 rows from those 280 with replacement, so some rows arrive
twice and about a third never arrive at all.

In [ ]:
one_tree(seed=0)

> **Question 1.** Change `seed=0` to `seed=2` and run the cell again. How much
> does the boundary move, and how much does the score?

<details>
<summary><b>Show the answer</b></summary>

The boundary is visibly a different shape and the fenced-off single points are
in different places, because a different third of the rows was left out. The
tree goes from 32 leaves to 29, and the share of rows held back it gets right
from 0.867 to 0.875. Nothing about the two arcs changed. All that changed was
which rows the tree happened to see.

</details>

## 3. The vote

In [ ]:
forest(trees=1)

> **Question 2.** Change `trees=1` to `trees=200`. What happens to the
> boundary, and what happens to the score?

<details>
<summary><b>Show the answer</b></summary>

The map goes from two flat colours to a gradient, and the pale band along the
middle is the ground the two arcs share. The score moves from 0.883 to 0.892,
about one row in the 120 held back.

The steadiness is the larger gain. Regrown twenty times from twenty seeds, a
one-tree forest scores between 0.792 and 0.883, a spread of 0.092. A 200-tree
forest scores between 0.892 and 0.908, a spread of 0.017. The standard
deviation falls from 0.0263 to 0.0046.

</details>

## 4. One point, ten opinions

In [ ]:
votes(row=4)

Ten trees, one row held back. Each bar is one tree, coloured by the arc it
picks. The star on the right is where that row sits.

Six trees say arc 1 and four say arc 0, so the forest says arc 1, and the row
truly belongs to arc 0. The row sits in the overlap, where the two arcs share
ground, and no vote taken over trees can recover what the measurements do not
distinguish.

> **Question 3.** Change `row=4` to `row=1`. How do the ten trees split, and
> does the forest get it right?

<details>
<summary><b>Show the answer</b></summary>

Ten out of ten, and the forest is right. Row 1 sits well inside its own arc,
where every tree draws the boundary on the same side of it. The split of the
vote is a reading of how firm the answer is: 10 to 0 is a point in clear
ground, 6 to 4 is a point in the overlap.

</details>

## 5. How many trees

In [ ]:
steadiness()

The dashed line is a single tree grown on all 280 rows, at 0.825. One
bootstrapped tree already stands above it at 0.883. The curve then moves
between 0.875 and 0.900 while the forest is small, and holds 0.892 from fifty
trees onward.

More trees never make a forest worse, and past a few dozen they buy almost
nothing. The count is chosen for the steadiness of the answer, and the cost is
linear in it.

## 6. Which measurement it leans on

Back to the 150 iris flowers, this time with all four measurements: the length
and width of the sepal and of the petal.

In [ ]:
importance(trees=200)

Each bar is the share of the total impurity removed by cuts on that column,
added over every tree. The two petal columns carry 0.873 between them, and the
width of the sepal carries 0.030.

> **Question 4.** Change `trees=200` to `trees=1`. What happens to the length
> of the sepal?

<details>
<summary><b>Show the answer</b></summary>

It drops to 0.000, because one tree found no cut it needed that column for.
The forest gives it 0.098: across 200 trees, on the resamples where the petal
columns were withheld at a cut, the length of the sepal did useful work. A
single tree reports which columns it used. A forest reports which columns carry
information.

The ranking is the trustworthy part. The values themselves are pulled upward
for columns with many distinct values, and two columns carrying the same
information split the credit between them.

</details>

## 7. The mathematics

Everything above works without this section.

**Why averaging helps.** Take $B$ trees, each an estimator with variance
$\sigma^2$, and let $\rho$ be the correlation between any two of them. The
average has variance

$$\rho \sigma^2 + \frac{1 - \rho}{B} \sigma^2$$

The second term goes to zero as trees are added, which is the settling of the
curve in section 5. The first term does not, and $\rho \sigma^2$ is the floor
no number of trees goes below. Lowering $\rho$ is what the two sources of
randomness are for: a resample of the rows for each tree, and a random subset
of the columns at each cut.

**Out of bag.** A bootstrap sample of $n$ rows drawn from $n$ leaves out any
given row with probability

$$\left(1 - \frac{1}{n}\right)^{n} \longrightarrow e^{-1} \approx 0.368$$

so about 36.8 per cent of the rows are absent from each tree. Scoring every row
on the trees that did not see it gives an estimate that costs no held-back
split at all.

**Importance.** The importance of column $j$ is the total impurity removed by
cuts on it, added over every node of every tree and divided by the number of
trees:

$$I_j = \frac{1}{B} \sum_{b=1}^{B} \sum_{\text{nodes } v \text{ cutting } j}
\frac{n_v}{n} \, \Delta(j, t_v)$$

In [ ]:
wood = RandomForestClassifier(n_estimators=200, oob_score=True,
                              random_state=0).fit(FIT_X, FIT_Y)
print("out of bag:    %.3f" % wood.oob_score_)
print("rows held back: %.3f" % wood.score(HELD_X, HELD_Y))
print("left out of a tree of 280 rows: %.3f"
      % (1 - 1 / 280) ** 280)

## What you can say now

- A forest grows many trees on resampled rows and answers by majority. The
  share of the vote is a gradient, and its width is where the trees disagreed.
- One tree is unsteady: change which rows it sees and the boundary moves. The
  standard deviation of the score over twenty regrowths falls from 0.0263 with
  one tree to 0.0046 with 200.
- The split of the vote says how firm an answer is. A point in the overlap
  between two classes splits the trees and stays wrong.
- Roughly 36.8 per cent of the rows are absent from any one tree, and scoring
  each row on the trees that missed it needs no held-back rows.
- Importance ranks the columns. The ranking is worth reading, the values less
  so.